In [21]:
import pandas as pd
import numpy as np

In [22]:
X_train = pd.read_csv("../data/X_train.csv")
y_train = pd.read_csv("../data/y_train.csv")

df = X_train.merge(y_train, on="id")

In [23]:
df.shape


(59400, 41)

In [24]:
df.describe()

,id,amount_tsh,gps_height,longitude,latitude,num_private,region_code,district_code,population,construction_year
count,59400.000000,59400.000000,59400.000000,59400.000000,5.940000e+04,59400.000000,59400.000000,59400.000000,59400.000000,59400.000000
mean,37115.131768,317.650385,668.297239,34.077427,-5.706033e+00,0.474141,15.297003,5.629747,179.909983,1300.652475
std,21453.128371,2997.574558,693.116350,6.567432,2.946019e+00,12.236230,17.587406,9.633649,471.482176,951.620547
min,0.000000,0.000000,-90.000000,0.000000,-1.164944e+01,0.000000,1.000000,0.000000,0.000000,0.000000
25%,18519.750000,0.000000,0.000000,33.090347,-8.540621e+00,0.000000,5.000000,2.000000,0.000000,0.000000
50%,37061.500000,0.000000,369.000000,34.908743,-5.021597e+00,0.000000,12.000000,3.000000,25.000000,1986.000000
75%,55656.500000,20.000000,1319.250000,37.178387,-3.326156e+00,0.000000,17.000000,5.000000,215.000000,2004.000000
max,74247.000000,350000.000000,2770.000000,40.345193,-2.000000e-08,1776.000000,99.000000,80.000000,30500.000000,2013.000000


In [25]:
print(df.columns)

Index(['id', 'amount_tsh', 'date_recorded', 'funder', 'gps_height',
       'installer', 'longitude', 'latitude', 'wpt_name', 'num_private',
       'basin', 'subvillage', 'region', 'region_code', 'district_code', 'lga',
       'ward', 'population', 'public_meeting', 'recorded_by',
       'scheme_management', 'scheme_name', 'permit', 'construction_year',
       'extraction_type', 'extraction_type_group', 'extraction_type_class',
       'management', 'management_group', 'payment', 'payment_type',
       'water_quality', 'quality_group', 'quantity', 'quantity_group',
       'source', 'source_type', 'source_class', 'waterpoint_type',
       'waterpoint_type_group', 'status_group'],
      dtype='str')


In [26]:
# Check missing values
print(df.isnull().sum())

id                           0
amount_tsh                   0
date_recorded                0
funder                    3637
gps_height                   0
installer                 3655
longitude                    0
latitude                     0
wpt_name                     2
num_private                  0
basin                        0
subvillage                 371
region                       0
region_code                  0
district_code                0
lga                          0
ward                         0
population                   0
public_meeting            3334
recorded_by                  0
scheme_management         3878
scheme_name              28810
permit                    3056
construction_year            0
extraction_type              0
extraction_type_group        0
extraction_type_class        0
management                   0
management_group             0
payment                      0
payment_type                 0
water_quality                0
quality_

In [ ]:
numeric_cols = ['gps_height', 'longitude', 'latitude', 
                'num_private', 'population', 'construction_year']

df[numeric_cols] = df[numeric_cols].replace(0, np.nan)


# Numeric columns → mean imputation
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].mean())

# Categorical columns → fill with "Unknown"
categorical_cols = df.select_dtypes(include=['object', 'str']).columns.tolist()
categorical_cols.remove('status_group')  # keep target untouched

for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")

# Replace '0' strings or numbers with 'Unknown' for categorical columns
for col in categorical_cols:
    df[col] = df[col].replace(['0', 0], 'special_unknown')

In [28]:
for col in categorical_cols:
    if (df[col] == '0').any() or (df[col] == 0).any():
        print(col, df[col].value_counts().loc[['0', 0]] if '0' in df[col].values or 0 in df[col].values else '')

In [9]:
# Check missing values
print(df.isnull().sum())

id                       0
amount_tsh               0
date_recorded            0
funder                   0
gps_height               0
installer                0
longitude                0
latitude                 0
wpt_name                 0
num_private              0
basin                    0
subvillage               0
region                   0
region_code              0
district_code            0
lga                      0
ward                     0
population               0
public_meeting           0
recorded_by              0
scheme_management        0
scheme_name              0
permit                   0
construction_year        0
extraction_type          0
extraction_type_group    0
extraction_type_class    0
management               0
management_group         0
payment                  0
payment_type             0
water_quality            0
quality_group            0
quantity                 0
quantity_group           0
source                   0
source_type              0
s

In [29]:
df.drop_duplicates(inplace=True)


In [30]:
# Convert date_recorded to datetime
df['date_recorded'] = pd.to_datetime(df['date_recorded'])
# Extract year, month, day
df['year_recorded'] = df['date_recorded'].dt.year



In [31]:
df['status_group'].value_counts()

status_group
functional                 32259
non functional             22824
functional needs repair     4317
Name: count, dtype: int64

In [32]:
for col in df.select_dtypes(include=['object', 'string']).columns:
    print(col, df[col].nunique())

funder 1896
installer 2145
wpt_name 37399
basin 9
subvillage 19288
region 21
lga 125
ward 2092
public_meeting 3
recorded_by 1
scheme_management 12
scheme_name 2696
permit 3
extraction_type 18
extraction_type_group 13
extraction_type_class 7
management 12
management_group 5
payment 7
payment_type 7
water_quality 8
quality_group 6
quantity 5
quantity_group 5
source 10
source_type 7
source_class 3
waterpoint_type 7
waterpoint_type_group 6
status_group 3


In [33]:
# Drop useless + extreme columns
df.drop(['recorded_by', 'wpt_name', 'subvillage', 'scheme_name', 'ward'], axis=1, inplace=True)

In [34]:
top_installers = df['installer'].value_counts().nlargest(10).index

df['installer_grouped'] = df['installer'].apply(
    lambda x: x if x in top_installers else 'Other'
)

In [35]:
top_funders = df['funder'].value_counts().nlargest(10).index

df['funder_grouped'] = df['funder'].apply(
    lambda x: x if x in top_funders else 'Other'
)

In [36]:
counts = df['lga'].value_counts()
df['lga_grouped'] = df['lga'].apply(lambda x: x if counts[x] > 150 else 'Other')

In [37]:
ct = pd.crosstab(df['lga_grouped'], df['status_group'])
ct_sorted = ct.sort_values(by='functional', ascending=False)
ct_sorted

status_group,functional,functional needs repair,non functional
lga_grouped,,,
Njombe,2007,94,402
Arusha Rural,875,48,329
Moshi Rural,733,119,399
Bagamoyo,730,2,265
Rungwe,676,161,269
...,...,...,...
Sikonge,49,0,121
Kiteto,45,28,120
Nanyumbu,42,0,116


In [38]:
missing_percent = df.isna().mean() * 100
missing_percent.sort_values(ascending=False)

id                       0.0
amount_tsh               0.0
date_recorded            0.0
funder                   0.0
gps_height               0.0
installer                0.0
longitude                0.0
latitude                 0.0
num_private              0.0
basin                    0.0
region                   0.0
region_code              0.0
district_code            0.0
lga                      0.0
population               0.0
public_meeting           0.0
scheme_management        0.0
permit                   0.0
construction_year        0.0
extraction_type          0.0
extraction_type_group    0.0
extraction_type_class    0.0
management               0.0
management_group         0.0
payment                  0.0
payment_type             0.0
water_quality            0.0
quality_group            0.0
quantity                 0.0
quantity_group           0.0
source                   0.0
source_type              0.0
source_class             0.0
waterpoint_type          0.0
waterpoint_typ

In [39]:
# Save cleaned DataFrame
df.to_csv("../data/cleaned_data.csv", index=False)